In [ ]:
#| default_exp onnx

# onnx

> `Model('yolo11n.onnx')` over ONNX Runtime, with preprocessing read off the graph.

`pip install 'anya[onnx]'`. Nothing here is called directly: `anya.core.Model` returns an
`OnnxModel` when the name ends in `.onnx` or the hub repo ships one.

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L, store_attr

from anya.core import Model, model_file

In [ ]:
#| hide
from fastcore.test import *
from anya.core import Preds, arrange
from tempfile import mkdtemp
FIX = Path('fixtures')

## Sessions

`providers` defaults to the fastest one ONNX Runtime reports as available, so a machine with CUDA or
Core ML uses it without being told, and a plain laptop falls back to the CPU provider.

In [ ]:
#| export
_ORT_DTYPES = {'tensor(float)': 'float32', 'tensor(float16)': 'float16', 'tensor(double)': 'float64',
               'tensor(uint8)': 'uint8', 'tensor(int8)': 'int8', 'tensor(int32)': 'int32',
               'tensor(int64)': 'int64', 'tensor(bool)': 'bool'}

PROVIDER_ORDER = ('TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CoreMLExecutionProvider',
                  'ROCMExecutionProvider', 'CPUExecutionProvider')

def best_providers() -> list:
    'The available execution providers, fastest first.'
    import onnxruntime as ort
    have = set(ort.get_available_providers())
    return [p for p in PROVIDER_ORDER if p in have] or ['CPUExecutionProvider']

def mk_session(path, providers=None, intra_threads:int=None, log:int=3):
    'An `onnxruntime.InferenceSession` over `path`, quiet by default.'
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.log_severity_level = log
    if intra_threads: so.intra_op_num_threads = intra_threads
    return ort.InferenceSession(str(path), sess_options=so, providers=providers or best_providers())

In [ ]:
#| hide
_s = mk_session(FIX/'tiny_cls.onnx')
test_eq([i.name for i in _s.get_inputs()], ['image'])
test_eq(best_providers()[-1], 'CPUExecutionProvider')

## OnnxModel

In [ ]:
#| export
class OnnxModel(Model):
    'An ONNX graph as a `Model`: shapes and dtypes come from the session, labels and norm from the caller or the hub.'
    _runtime = 'onnx'

    def __init__(self,
                 model=None,             # a path to an .onnx file, or a hub repo id
                 *,
                 runtime:str=None,       # ignored; `Model` has already dispatched on it
                 model_path=None,        # an explicit file, skipping resolution
                 file:str=None,          # which file to take from a repo that ships several
                 revision:str=None,
                 task:str=None,          # override the task guessed from the output shapes
                 labels=None,            # class names: a list, a labels.txt, or a config.json
                 norm=None,              # a NORMS name or an explicit (mean, std); the repo's config by default
                 size:tuple=None,        # the input size, for a graph with symbolic spatial axes
                 resize:str=None,        # 'stretch', 'letterbox', 'center_crop'
                 crop_pct:float=None,    # fraction of the short side 'center_crop' keeps
                 resample:int=None,      # PIL resample filter, 2 bilinear or 3 bicubic
                 prep=None,              # a fully built Prep, overriding everything above
                 topk:int=5, conf:float=0.25, iou:float=0.45,
                 providers=None,         # execution providers, fastest-available by default
                 sess=None,              # an already-built session to reuse
                 max_bs:int=16,          # cap on batching, when the graph allows any
                 **kw):
        model = self._setup(model, task=task, labels=labels, topk=topk, conf=conf, iou=iou)
        self.model_path = str(model_file(model, model_path, file=file, revision=revision))
        self._sess = sess or mk_session(self.model_path, providers=providers, **kw)
        self._read_spec()
        self._finish(task, prep, norm=norm, size=size, resize=resize, crop_pct=crop_pct, resample=resample)
        self._max_bs = self._batch_limit(max_bs)

    def _read_spec(self):
        'Input and output signatures, with ONNX Runtime type strings turned into numpy dtypes.'
        f = lambda o: AttrDict(name=o.name, shape=list(o.shape), dtype=_ORT_DTYPES.get(o.type, 'float32'))
        self.inputs, self.outputs = L(self._sess.get_inputs()).map(f), L(self._sess.get_outputs()).map(f)
        if len(self.inputs) > 1: raise ValueError(
            f'{Path(self.model_path).name} takes {len(self.inputs)} inputs '
            f'({", ".join(self.inputs.attrgot("name"))}); anya drives single-input vision graphs.')
        self.inp = self.inputs[0]

    def _batch_limit(self, mx:int) -> int:
        "A fixed batch axis is the limit; a symbolic one means `mx` is."
        b = self.inp.shape[0]
        return b if isinstance(b, int) and b > 0 else mx

    @property
    def spec(self) -> AttrDict:
        'What the graph declares, as plain dicts.'
        return AttrDict(inputs=list(self.inputs), outputs=list(self.outputs), providers=self._sess.get_providers())

    def _infer(self, x) -> list:
        if self._sess is None: raise RuntimeError('this model is closed')
        return self._sess.run(None, {self.inp.name: x.astype(self.inp.dtype, copy=False)})

## Running one

`tiny_cls.onnx` averages each channel and reads it as a class score, so a red picture is class 0 and
a green one is class 1. Small enough to be obvious, real enough to exercise the whole path.

In [ ]:
m = Model(FIX/'tiny_cls.onnx', labels=['red', 'green', 'blue', 'none'])
m

In [ ]:
green = np.zeros((40, 40, 3), np.uint8); green[..., 1] = 255
m(green)

In [ ]:
#| hide
test_eq(m.runtime, 'onnx'); test_eq(m.task, 'classify')
test_eq(m.prep.layout, 'nchw'); test_eq(m.prep.size, (8, 8))
test_eq(m(green).label, 'green')
red = np.zeros((40, 40, 3), np.uint8); red[..., 0] = 255
test_eq(m(red).label, 'red')
test_eq(m(red).preds[0]['index'], 0)
test_eq(len(m(red, topk=2).preds), 2)
test_eq(m.max_bs, 1)                      # the graph fixes its batch axis at 1

## Running a folder

`predict_all` takes the folder, batches to whatever the graph allows, and never lets one broken file
end the run.

In [ ]:
#| hide
from PIL import Image
_d = Path(mkdtemp())
for i, c in enumerate(['red', 'green', 'blue']):
    a = np.zeros((20, 20, 3), np.uint8); a[..., i] = 255
    Image.fromarray(a).save(_d/f'{c}_1.png')
(_d/'broken.png').write_bytes(b'not a png')

_ps = m.predict_all(_d)
test_eq(len(_ps), 4)
test_eq(_ps.counts(), {'blue': 1, 'green': 1, 'red': 1})
test_eq(len(_ps.failed), 1)
test_eq(_ps.failed[0]['src'].endswith('broken.png'), True)
test_fail(lambda: m.predict_all(_d, on_error='raise'))

# a folder in, a sorted folder out
_out = Path(mkdtemp())/'sorted'          # not under _d: a later run of the folder must not see them
_r = arrange(_ps.above(0.3), _out, how='copy', dry_run=False)
test_eq(sorted(p.name for p in _out.iterdir()), ['blue', 'green', 'red'])

## The other three tasks

Task detection reads the output signature, so a detector, a segmenter and an embedder all load
through the same call. No `task=` was passed to any of these.

In [ ]:
#| hide
_det = Model(FIX/'tiny_det.onnx', labels=['a','b','c'])
test_eq(_det.task, 'detect')
test_eq(_det.prep.resize, 'letterbox')       # detectors keep aspect ratio by default
test_eq(_det.prep.size, (100, 100))
_p = _det(np.zeros((50, 100, 3), np.uint8))
test_eq(len(_p.objects), 1); test_eq(_p.objects[0]['label'], 'b')
test_close(_p.objects[0]['box'], [40., 15., 60., 35.])     # un-letterboxed onto the 50x100 original

_seg = Model(FIX/'tiny_seg.onnx', labels=['red','green','blue'])
test_eq(_seg.task, 'segment')
_sp = _seg(green)
test_eq(_sp.label, 'green'); test_eq(_sp.classes[0]['frac'], 1.0)
test_eq(_sp['mask'].shape, (40, 40))         # the 8x8 output put back on the 40x40 picture it came from

_emb = Model(FIX/'tiny_emb.onnx')
test_eq(_emb.task, 'embed')                  # 128 features with no labels is not a 128-class model
test_eq(_emb.max_bs, 16)                     # a symbolic batch axis, so batching is allowed
_ep = _emb(green)
test_eq(_ep.vec.shape, (128,))
test_close(float(np.linalg.norm(_ep.vec)), 1.0)
test_eq(len(_emb.predict_all(_d)), 4)        # batched, and the broken file still reported

In [ ]:
#| hide
test_eq(_emb.spec.inputs[0]['shape'][0], 'batch')
_emb.close()
test_fail(lambda: _emb(green), contains='closed')

## A real one

Nothing above needs the network. This is the same call against a timm export, whose input signature
is `['batch_size', 'num_channels', 'height', 'width']`: no size, no channel count, only names. The
224 and the 0.875 crop come from `preprocessor_config.json`, and `evals/e2e.py` checks the resulting
tensor against timm's own eval transform.

In [ ]:
#| eval: false
m = Model('onnx-community/mobilenetv4_conv_small.e2400_r224_in1k')
print(m.inp.shape)      # ['batch_size', 'num_channels', 'height', 'width']
print(m.prep)           # Prep((224, 224), nchw, float32, resize=center_crop, crop_pct=0.875, resample=3)
m('two-cats.jpg')       # tabby, tabby cat (0.765)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()